In [875]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-07-03
Last modified on 2024-07-03
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, 
@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md
'''

'\nCreated on 2024-07-03\nLast modified on 2024-07-03\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener las relaciones entre técnicas, tácticas, grupos, software, \n@reference documentantion: https://github.com/mitre-attack/attack-stix-data/blob/master/USAGE.md\n'

**Índice de contenidos**

[Requerimientos]()

[Funciones]()

[Parámetros]()

[Ejecución principal]()


[1. Elementos generales]()

- [1.1. Generación de la matrix MITRE]()

- [1.2. Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada]()

- [1.3. Obtención de las TTP disponbles en Cyber Proof con regla de detección]()

[2. Tablas informativas]()

- [2.1. Técnicas]()

- [2.2. Tácticas]()

- [2.3. Data sources]()

- [2.4. Plataformas]()

- [2.5. Grupos]()

- [2.6. Software]()

[3. Relaciones]()

- [3.1. Relación técnicas - tácticas]()

- [3.2. Relación técnicas - data sources]()

- [3.3. Relación técnicas - plataformas]()

- [3.4. Relación técnicas - grupos]()

- [3.5. Relación técnicas - software]()

- [3.6. Relación grupos - software]()

[4. Disponibilidad de reglas por tipología]()

- [4.1. Reglas disponibles por técnica]()

- [4.2. Reglas disponibles por técnica y táctica]()

- [4.3. Reglas disponibles por técnica y data source]()

- [4.4. Reglas disponibles por técnica y plataforma]()

- [4.5. Reglas disponibles por técnica y grupo]()

- [4.6. Reglas disponibles por técnica y software]()

#### **Requerimientos**

In [876]:
from stix2 import Filter, MemoryStore
import stix2
import requests

import os
import pandas as pd

import datetime
import csv

#### **Funciones**

**Generales**

In [877]:
def create_output_folder(path):
    '''
    Función encargada para crear el directorio facilitado en caso de no existir previamente.
    '''
    if not os.path.exists(path):
        os.makedirs(path)

In [878]:
def get_list_of_files_sub(dir_name):
    '''
    Función encargada de retornar una lista de archivos ubicados en la ruta facilitada así como en los subdirectorios disponibles.
    '''
    listOfFile = os.listdir(dir_name)
    allFiles = list()
    for entry in listOfFile:
        fullPath = os.path.join(dir_name, entry)
        if os.path.isdir(fullPath):
            allFiles = allFiles + get_list_of_files_sub(fullPath)
        else:
            allFiles.append(fullPath)
    return allFiles

In [879]:
def get_unique_ttps_generated(paths):
    '''
    Función encargada de retornar una lista de id de TTP's únicas generadas. Esta función recibirá una ruta generada previamente en la estructura de carpetas de guardado. 
    '''
    sub_folders = set()
    for path in paths:
        dir, file = os.path.split(path)
        last_sub_folder = os.path.basename(dir).strip()
        sub_folders.add(last_sub_folder)
    return list(sub_folders)

In [880]:
def unique_list(series):
    '''
    Función para devolver sólo ítems únicos a partir de una lista.
    '''
    return list(set(series))

In [881]:
def get_data_from_branch(matrix):
    '''
    Función encargada de peticionar a la url de GitHub donde está publicada la última versión MITRE de la información en formato stix2. Retorna el objeto que contiene toda la información de MITRE.
    '''
    url = f"https://raw.githubusercontent.com/mitre/cti/master/{matrix}-attack/{matrix}-attack.json"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Verificar si la solicitud fue exitosa
        stix_json = response.json()
        
        if "objects" in stix_json:
            return MemoryStore(stix_data=stix_json["objects"])
        else:
            raise ValueError("JSON no contiene la clave 'objects'")
            
    except requests.RequestException as e:
        print(f"Error al obtener datos de {matrix}-attack: {e}")
        return None

In [882]:
def save_df_as_csv(df, name, matrix, aux_folder=''):
    '''
    Función encargada de guardar un dataframe pasado como argumento como csv.
    '''
    now = datetime.datetime.now()
    if not os.path.exists(os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder)):
        os.makedirs(os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder))
    # file_to_save = os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder, (name+ '_'+ now.strftime('%d%m%Y_%H%M')+'h.csv'))
    file_to_save = os.path.join(os.getcwd(), 'outputs/stix2',  matrix, aux_folder, (name + '.csv'))
    df.to_csv(file_to_save, sep=';', encoding='utf8', index=False, quoting=csv.QUOTE_NONNUMERIC)
    # print('Archivo guardado correctamente '+ name + '_' +now.strftime('%d%m%Y_%H%M')+'h.csv')
    print('Archivo guardado correctamente '+ (name + '.csv'))

In [883]:
def get_list_techniques_from_stix2(src, include="both"):
    '''
    Función encargada de la obtención de la lista de técnicas de los datos facilitados. Por defecto se retornan tanto técnicas como subtécnicas. Retorna una lista de strings con el id correspondiente a cada técnica.
    '''
    if include == "techniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', False)
        ])
    elif include == "subtechniques":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern'),
            Filter('x_mitre_is_subtechnique', '=', True)
        ])
    elif include == "both":
        query_results = src.query([
            Filter('type', '=', 'attack-pattern')
        ])
    else:
        raise RuntimeError("Unknown option %s!" % include)
    
    if isinstance(query_results, stix2.datastore.memory.MemoryStore):
        query_results = query_results.query()
        stix2_df = pd.DataFrame(query_results)
    elif isinstance(query_results[0], stix2.v20.sdo.AttackPattern):
        stix2_df = pd.DataFrame(query_results)

    stix2_df['technique_id'] = stix2_df['external_references'].apply(lambda refs: refs[0].external_id if refs else None)
    stix2_df = stix2_df[(stix2_df['revoked']!=True)&(stix2_df['x_mitre_deprecated']!=True)]
    stix2_df['technique_id'] = stix2_df['technique_id'].str.upper()
    techniques = sorted(stix2_df['technique_id'].drop_duplicates(), key=len, reverse=True)

    return techniques

In [884]:
def get_CP_ttps_with_rules(mitre_matrix_str, way='file'):
    '''
    Función encargada de obtener el listado de TTP's para las que desde CP disponemos de reglas de detección. Para la ejecutarla correctamente debe de haberse ejecutado previamente el código del bloque get_rules_and_classify_by_ttp. Hay dos formas de evaluar las técnicas disponibles y son gestionadas por el parámetro "way". Por defecto el modo es "file", lo que nos indica que se va a evaluar uno de los ficheros generados de resumen de reglas por TTP clasificadas. En el caso de que el parámetro "way" recoja el valor "path", lo que hará es identificar las reglas mediante la exploración de subdirectorios. 
    '''
    try:
        main_path = os.path.join(os.path.dirname(os.getcwd()), 'get_rules_and_classify_by_ttp', 'outputs', mitre_matrix_str)
        
        if not os.path.exists(main_path):
            raise ValueError(f'No existe la path: {main_path}')
        
        if way == 'file':
            file_path = os.path.join(main_path, f'{mitre_matrix_str}-ttp_all_classified_rules.csv')  # Corrección aquí
            file = pd.read_csv(file_path, sep=';')
            file = file[file['ttp'] != 'T0000']  # Filtramos la técnica ficticia donde metemos las reglas que no han sido mapeadas
            CP_techniques = file['ttp'].unique().tolist()  # Convertimos en lista de items únicos la columna que informa de la ttp
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)  # Ordenamos por longitud de caracteres para que cuando sea utilizada esta lista primero se evalúen las subtécnicas.

        elif way == 'path':
            CP_techniques = get_unique_ttps_generated(get_list_of_files_sub(main_path))
            CP_techniques = sorted(CP_techniques, key=len, reverse=True)
            CP_techniques = [ttp for ttp in CP_techniques if ttp.startswith('T')]
            CP_techniques = [ttp for ttp in CP_techniques if ttp != 'T0000']

        else:
            CP_techniques = []
            
    except ValueError as e:
        print(f'{e}')
        CP_techniques = []
    
    return CP_techniques

In [885]:
def get_CP_and_NOCP_techniques_from_df(cp_techniques_list, df):
    '''
    Función encargada de filtrar un df dado el cual contiene el campo techniques_ID por las técnicas disponibles en CP (estas son facilitadas como parámetro en forma de lista). 
    '''
    cp_df = pd.DataFrame()
    nocp_df = pd.DataFrame() 
    try:
        cp_df = df[df['technique_ID'].isin(cp_techniques_list)]
        nocp_df = df[~df['technique_ID'].isin(cp_techniques_list)]
    except:
        print('No se ha podido ejecutar la operación.')
    return cp_df, nocp_df

**Obtención de información**

In [886]:
def get_techniques_information(matrix_store, cp_techniques_list, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a la técnicas (ID, nombre, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    techniques_data = []
    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    for tech in techniques:
        deprecated = False
        revoked = False
        description = ''
        if 'x_mitre_deprecated' in tech: 
            deprecated = tech['x_mitre_deprecated']
        if 'revoked' in tech: 
            revoked = tech['revoked']
        if 'description' in tech: 
            description = tech['description']
        techniques_data.append({
            "technique_ID": tech['external_references'][0]['external_id'],
            "technique": tech['name'],
            "technique_url": tech['external_references'][0]['url'],
            "technique_description":description,
            "technique_deprecated": deprecated,
            "technique_revoked": revoked,
            "matrix_domains": tech['x_mitre_domains'] # pendiente chequear
        })
        
    MITRE_techniques_df = pd.DataFrame(techniques_data)

    if revoked_deprecated:
        MITRE_techniques_df = MITRE_techniques_df[(MITRE_techniques_df['technique_deprecated']!=True)&(MITRE_techniques_df['technique_revoked']!=True)]

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_df, NOCP_techniques_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_df)

    return MITRE_techniques_df, CP_techniques_df, NOCP_techniques_df

In [887]:
def get_tactics_information(matrix_store):
    '''
    Función que retorna el dataframe con la información relativa a la tácticas (ID, nombre, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    tactics_data = []
    tactics = matrix_store.query([Filter('type', '=', 'x-mitre-tactic')])
    for tact in tactics:
        tactics_data.append({
            "tactic_ID": tact['external_references'][0]['external_id'],
            "tactic": tact['name'],
            "tactic_url": tact['external_references'][0]['url'],
            "tactic_description":tact['description'],
            "matrix_domains": tact['x_mitre_domains'] # pendiente chequear
        })
    MITRE_tactics_df = pd.DataFrame(tactics_data)
    return MITRE_tactics_df

In [888]:
def get_datasources_information(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a los data sources (ID, nombre, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    datasoruces_data = []
    data_sources = matrix_store.query([Filter('type', '=', 'x-mitre-data-source')])
    for ds in data_sources:
        deprecated = False
        revoked = False
        if 'x_mitre_deprecated' in ds: 
            deprecated = ds['x_mitre_deprecated']
        if 'revoked' in ds: 
            revoked = ds['revoked']

        datasoruces_data.append({
            "data_source_ID": ds['external_references'][0]['external_id'],
            "data_source": ds['name'],
            "data_source_url": ds['external_references'][0]['url'],
            "data_source_description":ds['description'],
            "data_source_deprecated": deprecated,
            "data_source_revoked": revoked,
            "matrix_domains": ds['x_mitre_domains'] # pendiente chequear
        })

    MITRE_datasources_df = pd.DataFrame(datasoruces_data)
    if revoked_deprecated:
        MITRE_datasources_df = MITRE_datasources_df[(MITRE_datasources_df['data_source_deprecated']!=True)&(MITRE_datasources_df['data_source_revoked']!=True)]
    return MITRE_datasources_df

In [889]:
def get_platforms_information(matrix_store):
    '''
    Función que retorna el dataframe con la información relativa a las plataformas. Únicamente está conformado por una columna con el nombre de la misma ya que las plataformas no se obtienen de una tipología propia de la matriz, se obtienen como parámetro de las técnicas.
    '''
    platforms_from_tech = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    platforms_data = []
    for technique in platforms_from_tech:
        platform_name = 'None'
        if 'x_mitre_data_sources' in technique: 
            try: 
                platform_name = technique['x_mitre_platforms']
            except:
                platform_name = 'None'

        platforms_data.append({
            "platform": platform_name,
        })
    # Generamos df a partir de los datos recopilados
    MITRE_platforms_df = pd.DataFrame(platforms_data)
    MITRE_platforms_df = MITRE_platforms_df.explode('platform')
    MITRE_platforms_df = MITRE_platforms_df.drop_duplicates()
    MITRE_platforms_df = MITRE_platforms_df.sort_values(by='platform')
    MITRE_platforms_df = MITRE_platforms_df.reset_index(drop=True)
    return MITRE_platforms_df

In [890]:
def get_groups_information(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a los grupos (ID, nombre, alias, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])
    groups_data = []
    for group in groups:
        aliases = ''
        description =''
        deprecated = False
        revoked = False
        if 'aliases' in group: 
            aliases = group['aliases']
        if 'description' in group: 
            description = group['description']
        if 'x_mitre_deprecated' in group: 
            deprecated = group['x_mitre_deprecated']
        if 'revoked' in group: 
            revoked = group['revoked']

        groups_data.append({
            "group_ID": group['external_references'][0]['external_id'],
            "group": group['name'],
            "group_aliases": aliases,
            "group_url": group['external_references'][0]['url'],
            "group_description": description,
            "group_deprecated": deprecated,
            "group_revoked": revoked,
            "matrix_domains": group['x_mitre_domains']
        })


    # Generamos df a partir de los datos recopilados
    MITRE_groups_df = pd.DataFrame(groups_data)
    if revoked_deprecated:
        MITRE_groups_df = MITRE_groups_df[(MITRE_groups_df['group_deprecated']!=True)&(MITRE_groups_df['group_revoked']!=True)]

    MITRE_groups_df = MITRE_groups_df.sort_values(by='group_ID')
    MITRE_groups_df = MITRE_groups_df.reset_index(drop=True)
    return MITRE_groups_df

In [891]:
def get_software_information(matrix_store, revoked_deprecated=True):
    '''
    Función que retorna el dataframe con la información relativa a los tipos de software (ID, nombre, alias, url, descripción, estado deprecado, estado revocado, dominios MITRE).
    '''
    malware = matrix_store.query([Filter('type', '=', 'malware')])
    tool = matrix_store.query([Filter('type', '=', 'tool')])
    software = malware + tool
    software_data = []
    for sw in software:
        description =''
        deprecated = False
        revoked = False
        if 'aliases' in sw: 
            aliases = sw['aliases']
        if 'description' in sw: 
            description = sw['description']
        if 'x_mitre_deprecated' in sw: 
            deprecated = sw['x_mitre_deprecated']
        if 'revoked' in sw: 
            revoked = sw['revoked']

        software_data.append({
            "software_ID": sw['external_references'][0]['external_id'],
            "software": sw['name'],
            "software_type": sw['type'],
            "software_url": sw['external_references'][0]['url'],
            "software_description": description,
            "software_deprecated": deprecated,
            "software_revoked": revoked,
            "matrix_domains": sw['x_mitre_domains']
        })


    # Generamos df a partir de los datos recopilados
    MITRE_software_df = pd.DataFrame(software_data)
    revoked_deprecated = True
    if revoked_deprecated:
        MITRE_software_df = MITRE_software_df[(MITRE_software_df['software_deprecated']!=True)&(MITRE_software_df['software_revoked']!=True)]

    MITRE_software_df = MITRE_software_df.sort_values(by='software_ID')
    MITRE_software_df = MITRE_software_df.reset_index(drop=True)
    return MITRE_software_df

**Generación de relaciones**

In [892]:
def get_techniques_tactics_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y tácticas (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    # Obtenemos las tácticas de la matriz
    tactics = matrix_store.query([
        Filter('type', '=', 'x-mitre-tactic')
    ])
    
    # Mediante diccionarios mapeamos la información que necesitamos
    technique_dict = {
        tech['external_references'][0]['external_id']: {
            'name': tech['name'],
            'deprecated': tech.get('x_mitre_deprecated', False),
            'revoked': tech.get('revoked', False)
        }
        for tech in techniques
        if 'external_references' in tech and len(tech['external_references']) > 0
    }
    tactic_dict = {
        tac['external_references'][0]['external_id']: tac['name']
        for tac in tactics
        if 'external_references' in tac and len(tac['external_references']) > 0
    }
    # Lista para almacenar las relaciones
    data = []

    # Recorrer todas las técnicas para encontrar sus relaciones con tácticas
    for technique in techniques:
        if 'kill_chain_phases' in technique:
            for phase in technique['kill_chain_phases']:
                # Cada fase representa una relación técnica-táctica
                tactic_shortname = phase['phase_name']
                external_id = technique['external_references'][0]['external_id'] if 'external_references' in technique and len(technique['external_references']) > 0 else None
                
                # Buscar el ID externo de la táctica correspondiente
                tactic_id = next((tac['external_references'][0]['external_id'] for tac in tactics if tac['x_mitre_shortname'] == tactic_shortname), None)
                # Si no hubiera ninguna relación
                if not external_id or not tactic_id:
                    continue
                # Agregamos la relación a la lista
                data.append({
                    "technique_ID": external_id,
                    "technique": technique_dict[external_id]['name'],
                    "tactic_ID": tactic_id,
                    "tactic": tactic_dict[tactic_id],
                    "technique_deprecated": technique_dict[external_id]['deprecated'],
                    "technique_revoked": technique_dict[external_id]['revoked']
                })
    
    # Creamos el dataframe final
    techniques_tactics_df = pd.DataFrame(data)
    #Filtramos técnicas deprecadas
    techniques_tactics_NN_df = techniques_tactics_df[(techniques_tactics_df['technique_deprecated']!=True)&(techniques_tactics_df['technique_revoked']!=True)]

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_tactics_NN_df = techniques_tactics_NN_df[['technique_ID', 'technique', 'tactic_ID', 'tactic']] # Filtramos las columnas que necesitamos
    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_tactics_NN_df)


    # Para finalizar agregamos por técnica y táctica con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y tácticas en una relación 1:N
    MITRE_technique_tactics_1N_df = MITRE_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    CP_technique_tactics_1N_df = CP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })
    NOCP_technique_tactics_1N_df = NOCP_techniques_tactics_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'tactic_ID': unique_list,
    'tactic': unique_list
    })


    # Generamos el df compuesto por tecnicas y tácticas en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_tactics_NN_df.groupby('tactic_ID', as_index=False).agg({
    'tactic':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [893]:
def get_techniques_datasources_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y data sources (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2.Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_ds_name = ''
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique: 
            ttp_ds_name = technique['x_mitre_data_sources']
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "data_source": ttp_ds_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)
    # Transformaciones de la tabla de relaciones de técnicas
    techniques_df = techniques_df.explode('data_source')
    techniques_df['data_source'] = techniques_df['data_source'].str.split(':').str[0] # El datasource al que aplica cada 
    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'data_source']]

    # Obtenemos los data sources de la matriz
    data_sources = matrix_store.query([
        Filter('type', '=', 'x-mitre-data-source')
    ])

    ds_data = []

    for data_source in data_sources:
        ds_id = ''
        ds_name = ''
        ds_revoked = ''
        ds_deprecated = ''
        if 'external_references' in data_source:
            ds_id = data_source['external_references'][0]['external_id']
        if 'name' in data_source:
            ds_name = data_source['name']
        if 'revoked' in data_source:
            ds_revoked = data_source['revoked']
        if 'x_mitre_deprecated' in data_source:
            ds_deprecated = data_source['x_mitre_deprecated']

        ds_data.append({
            "data_source_ID": ds_id,
            "data_source": ds_name,
            "data_source_deprecated": ds_deprecated,
            "data_source_revoked": ds_revoked
        })
    # Generamos df a partir de los datos recopilados
    data_sources_df = pd.DataFrame(ds_data)
    # Transformaciones de la tabla de relaciones de data sources
    data_sources_df = data_sources_df[(data_sources_df['data_source_deprecated']!=True)&(data_sources_df['data_source_revoked']!=True)]
    data_sources_df = data_sources_df[['data_source_ID', 'data_source']] 
    data_sources_df = data_sources_df.sort_values(by='data_source_ID')

    # Unimos las tablas
    techniques_data_sources_df = pd.merge(techniques_df, data_sources_df, on='data_source', how='left')
    techniques_data_sources_df = techniques_data_sources_df.drop_duplicates()

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y data sources en una relación N:N
    MITRE_techniques_datasources_NN_df = techniques_data_sources_df[['technique_ID', 'technique', 'data_source_ID', 'data_source']]
    MITRE_techniques_datasources_NN_df = MITRE_techniques_datasources_NN_df.sort_values(by='technique_ID').reset_index(drop=True)
    MITRE_techniques_datasources_NN_df

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_datasources_NN_df)


    # Para finalizar agregamos por técnica y data sources con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y data sources en una relación 1:N
    MITRE_technique_datasources_1N_df = MITRE_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    CP_technique_datasources_1N_df = CP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })
    NOCP_technique_datasources_1N_df = NOCP_techniques_datasources_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'data_source_ID': unique_list,
    'data_source': unique_list
    })

    # Generamos el df compuesto por tecnicas y data sources en una relación N:1
    MITRE_techniques_tactic_N1_df = MITRE_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_tactic_N1_df = CP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_tactic_N1_df = NOCP_techniques_datasources_NN_df.groupby('data_source_ID', as_index=False).agg({
    'data_source':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df

In [894]:
def get_techniques_platforms_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y plataformas (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la generación de la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Obtenemos las técnicas de la matriz
    techniques = matrix_store.query([
        Filter('type', '=', 'attack-pattern')
    ])
    techniques_data = []
    for technique in techniques:
        ttp_name = ''
        ttp_id = ''
        ttp_platform_name = 'None'
        ttp_revoked = ''
        ttp_deprecated = ''
        if 'external_references' in technique:
            ttp_id = technique['external_references'][0]['external_id']
        if 'name' in technique:
            ttp_name = technique['name']
        if 'x_mitre_data_sources' in technique:
            try: 
                ttp_platform_name = technique['x_mitre_platforms']
            except:
                ttp_platform_name = 'None'
        if 'revoked' in technique: 
            ttp_revoked = technique['revoked']
        if 'x_mitre_deprecated' in technique: 
            ttp_deprecated = technique['x_mitre_deprecated']

        techniques_data.append({
            "technique_ID": ttp_id,
            "technique": ttp_name,
            "platform": ttp_platform_name,
            "technique_deprecated": ttp_deprecated,
            "technique_revoked": ttp_revoked
        })
    # Generamos df a partir de los datos recopilados
    techniques_df = pd.DataFrame(techniques_data)

    techniques_df = techniques_df[(techniques_df['technique_deprecated']!=True)&(techniques_df['technique_revoked']!=True)]
    techniques_df = techniques_df[['technique_ID', 'technique', 'platform']]
    techniques_df = techniques_df.explode('platform')
    MITRE_techniques_platforms_NN_df = techniques_df.drop_duplicates()
    MITRE_techniques_platforms_NN_df = MITRE_techniques_platforms_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_platforms_NN_df)


    # Para finalizar agregamos por técnica y plataformas con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y plataformas en una relación 1:N
    MITRE_technique_platforms_1N_df = MITRE_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    CP_technique_platforms_1N_df = CP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })
    NOCP_technique_platforms_1N_df = NOCP_techniques_platforms_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'platform': unique_list
    })

    # Generamos el df compuesto por tecnicas y plataformas en una relación N:1
    MITRE_techniques_platform_N1_df = MITRE_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_platform_N1_df = CP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_platform_N1_df = NOCP_techniques_platforms_NN_df.groupby('platform', as_index=False).agg({
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df

In [895]:
def get_techniques_groups_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y grupos (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Verificamos que no haya habido algun error en la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Filtramos los objetos de tipo intrusion-set (grupos), attack-pattern (técnicas) y relationship (relaciones)
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])

    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # En 2 diccionarios mapeamos los IDs y nombres
    group_id_map = {g['id']: next((ref['external_id'] for ref in g['external_references'] if ref['source_name'] == 'mitre-attack'), None) for g in groups}
    technique_id_map = {t['id']: {
            'technique_id': next((ref['external_id'] for ref in t['external_references'] if ref['source_name'] == 'mitre-attack'), None),
            'technique_name': t['name']
        } for t in techniques}

    # Creamos un diccionario para mapear los grupos y las técnicas que usan mediante las relaciones
    group_techniques = []

    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'].startswith('intrusion-set') and rel['target_ref'].startswith('attack-pattern'):
            group_id = rel['source_ref']
            technique_id = rel['target_ref']

            group_abbrev_id = group_id_map.get(group_id)
            technique_info = technique_id_map.get(technique_id)
            group_name = next((g['name'] for g in groups if g['id'] == group_id), None)

            if group_abbrev_id and technique_info and group_name:
                group_techniques.append({
                    'group_ID': group_abbrev_id,
                    'group': group_name,
                    'technique_ID': technique_info['technique_id'],
                    'technique': technique_info['technique_name']
                })

    # Crear un dataframe con los resultados
    df = pd.DataFrame(group_techniques, columns=['group_ID', 'group', 'technique_ID', 'technique'])

    MITRE_techniques_groups_NN_df = df.drop_duplicates()
    MITRE_techniques_groups_NN_df = MITRE_techniques_groups_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_groups_NN_df)


    # Para finalizar agregamos por técnica y grupos con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y grupos en una relación 1:N
    MITRE_technique_groups_1N_df = MITRE_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })
    CP_technique_groups_1N_df = CP_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })
    NOCP_technique_groups_1N_df = NOCP_techniques_groups_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })

    # Generamos el df compuesto por tecnicas y plataformas en una relación N:1
    MITRE_techniques_group_N1_df = MITRE_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_group_N1_df = CP_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_group_N1_df = NOCP_techniques_groups_NN_df.groupby('group_ID', as_index=False).agg({
    'group': unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_groups_NN_df, CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df, MITRE_technique_groups_1N_df, CP_technique_groups_1N_df, NOCP_technique_groups_1N_df, MITRE_techniques_group_N1_df, CP_techniques_group_N1_df, NOCP_techniques_group_N1_df

In [896]:
def get_techniques_software_relationships(matrix_store, cp_techniques_list):
    '''
    Función que retorna las relaciones entre técnicas y software (N:N, 1:N y N:1), también devuelve estas tablas como dataframes de pandas filtrados por: todos los disponibles en MITRE, los disponibles en CyberProof y los NO disponibles en CyberProof. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna 9 df, 3 para las posibles relaciones 1:N, N:N y N:1, y 3 opciones para MITRE (todas las técnicas), CP (sólo técnicas disponibles en CyberProof) y NOCP (sólo técnicas ausentes en CyberProof).
    '''
    # Filtrar técnicas y software
    techniques = matrix_store.query([Filter('type', '=', 'attack-pattern')])
    software = matrix_store.query([Filter('type', '=', 'malware')]) + matrix_store.query([Filter('type', '=', 'tool')])
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # Crear diccionarios para los IDs y nombres de técnicas y software
    technique_id_to_name = {}
    software_id_to_name = {}

    for tech in techniques:
        for ref in tech.get('external_references', []):
            if 'external_id' in ref:
                technique_id_to_name[tech['id']] = {
                    'external_id': ref['external_id'],
                    'name': tech['name'],
                    'x_mitre_deprecated': tech.get('x_mitre_deprecated', False),
                    'revoked': tech.get('revoked', False)
                }
                break

    for sw in software:
        for ref in sw.get('external_references', []):
            if 'external_id' in ref:
                software_id_to_name[sw['id']] = {
                    'external_id': ref['external_id'],
                    'name': sw['name']
                }
                break

    # Extraer las relaciones entre técnicas y software
    techniques_software_data = []
    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'] in software_id_to_name and rel['target_ref'] in technique_id_to_name:
            software_info = software_id_to_name[rel['source_ref']]
            technique_info = technique_id_to_name[rel['target_ref']]
            techniques_software_data.append({
                'software_ID': software_info['external_id'],
                'software': software_info['name'],
                'technique_ID': technique_info['external_id'],
                'technique': technique_info['name'],
                'technique_deprecated': technique_info['x_mitre_deprecated'],
                'technique_revoked': technique_info['revoked']
            })

    techniques_software_df = pd.DataFrame(techniques_software_data)

    # Transformaciones de la tabla de relaciones de data sources
    techniques_software_df = techniques_software_df[(techniques_software_df['technique_deprecated']!=True)&(techniques_software_df['technique_revoked']!=True)]
    techniques_software_df = techniques_software_df.drop_duplicates()

    # Generamos el que finalmente será nuestro df compuesto por tecnicas y tácticas en una relación N:N
    MITRE_techniques_software_NN_df = techniques_software_df[['technique_ID', 'technique', 'software_ID', 'software']]
    MITRE_techniques_software_NN_df = MITRE_techniques_software_NN_df.sort_values(by='technique_ID').reset_index(drop=True)

    # A continuación comenzamos por filtrar la tabla fundamental por las técnicas disponibles y no disponibles en CP
    CP_techniques_software_NN_df, NOCP_techniques_software_NN_df = get_CP_and_NOCP_techniques_from_df(cp_techniques_list, MITRE_techniques_software_NN_df)

    # Para finalizar agregamos por técnica y software con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por técnica y software en una relación 1:N
    MITRE_technique_software_1N_df = MITRE_techniques_software_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })
    CP_technique_software_1N_df = CP_techniques_software_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })
    NOCP_technique_software_1N_df = NOCP_techniques_software_NN_df.groupby('technique_ID', as_index=False).agg({
    'technique':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })

    # Generamos el df compuesto por tecnicas y software en una relación N:1
    MITRE_techniques_software_N1_df = MITRE_techniques_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    CP_techniques_software_N1_df = CP_techniques_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })
    NOCP_techniques_software_N1_df = NOCP_techniques_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'technique_ID': unique_list,
    'technique': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_techniques_software_NN_df, CP_techniques_software_NN_df, NOCP_techniques_software_NN_df, MITRE_technique_software_1N_df, CP_technique_software_1N_df, NOCP_technique_software_1N_df, MITRE_techniques_software_N1_df, CP_techniques_software_N1_df, NOCP_techniques_software_N1_df

In [897]:
def get_groups_software_relationships(matrix_store):
    '''
    Función que retorna las relaciones entre grupos y software (N:N, 1:N y N:1). A diferencia de anteriores relaciones, en este caso no se puede filtrar por lista de técnicas disponibles. Requiere la matriz MITRE (enterprise, ICS o Mobile) en formato MemoryStore de stix2. Retorna únicamente 3 df para las posibles relaciones 1:N, N:N y N:1 ya que en esta relación no hay disponibilidad de filtrar por técnica.
    '''
    # Verificamos que no haya habido algun error en la matriz
    if matrix_store is None:
        raise ValueError("El MemoryStore está vacío o no se ha inicializado correctamente.")

    # Filtramos los objetos de tipo intrusion-set (grupos), tool y malware (software) y relationship (relaciones)
    groups = matrix_store.query([Filter('type', '=', 'intrusion-set')])
    malware = matrix_store.query([Filter('type', '=', 'malware')])
    tool = matrix_store.query([Filter('type', '=', 'tool')])
    software = malware + tool
    relationships = matrix_store.query([Filter('type', '=', 'relationship')])

    # Creamos diccionarios para mapear los IDs y nombres
    group_id_map = {g['id']: next((ref['external_id'] for ref in g['external_references'] if ref['source_name'] == 'mitre-attack'), None) for g in groups}
    software_id_map = {s['id']: {
            'software_ID': next((ref['external_id'] for ref in s['external_references'] if ref['source_name'] == 'mitre-attack'), None),
            'software': s['name']
        } for s in software}

    # Creamos un diccionario para mapear los grupos y el software que usan
    group_software = []

    for rel in relationships:
        if rel['relationship_type'] == 'uses' and rel['source_ref'].startswith('intrusion-set') and rel['target_ref'].startswith(('tool--', 'malware--')):
            group_id = rel['source_ref']
            software_id = rel['target_ref']

            group_abbrev_id = group_id_map.get(group_id)
            software_info = software_id_map.get(software_id)
            group_name = next((g['name'] for g in groups if g['id'] == group_id), None)

            if group_abbrev_id and software_info and group_name:
                group_software.append({
                    'group_ID': group_abbrev_id,
                    'group': group_name,
                    'software_ID': software_info['software_ID'],
                    'software': software_info['software']
                })

    # Dataframe con los resultados
    MITRE_groups_software_NN_df = pd.DataFrame(group_software, columns=['group_ID', 'group', 'software_ID', 'software'])

    # Para finalizar agregamos por grupo y software con el objetivo de poder ofrecer otras tablas valiosas de cara a realizar consultas
    # Generamos el df compuesto por grupo y software's en una relación 1:N
    MITRE_group_software_1N_df = MITRE_groups_software_NN_df.groupby('group_ID', as_index=False).agg({
    'group':unique_list,
    'software_ID': unique_list,
    'software': unique_list
    })

    # Generamos el df compuesto por grupos y software en una relación N:1
    MITRE_groups_software_N1_df = MITRE_groups_software_NN_df.groupby('software_ID', as_index=False).agg({
    'software':unique_list,
    'group_ID': unique_list,
    'group': unique_list
    })

    print('Dataframes generados correctamente!')

    return MITRE_groups_software_NN_df, MITRE_group_software_1N_df, MITRE_groups_software_N1_df

**Disponibilidad de reglas**

In [898]:
def get_availability_rules_by_technique(mitre_matrix_str):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica. Devuelve una primera tabla idéntica a la generada en el resumen generado en 'get_rules_and_classify_by_ttp/{matrix}-classified_rules.csv'. Y por otro lado genera una tabla con el agregado del número de reglas por técnica disponible que será requerida en posteriores cruces.
    '''
    try:
        main_path = os.path.join(os.path.dirname(os.getcwd()), 'get_rules_and_classify_by_ttp', 'outputs', mitre_matrix_str)
        CP_detail_rules_techniques = pd.DataFrame()

        if not os.path.exists(main_path):
            raise ValueError(f'No existe el fichero requerido: {main_path}, por favor ejecute el bloque get_rules_and_classify_by_ttp')

        file_path = os.path.join(main_path, f'{mitre_matrix_str}-classified_rules.csv')  # Corrección aquí
        CP_detail_rules_techniques = pd.read_csv(file_path, sep=';')
        CP_detail_rules_techniques = CP_detail_rules_techniques[CP_detail_rules_techniques['ttp'] != 'T0000']  # Filtramos la técnica ficticia donde metemos las reglas que no han sido mapeadas
        CP_detail_rules_techniques = CP_detail_rules_techniques.sort_values(by='ttp')
        CP_detail_rules_techniques = CP_detail_rules_techniques.reset_index(drop=True)

        CP_agg_rules_techniques = CP_detail_rules_techniques[['ttp', 'rule', 'matrix']]
        CP_agg_rules_techniques = CP_agg_rules_techniques.groupby('ttp').agg(
            rules=('rule', 'count'),
            matrix=('matrix', 'first')
        ).sort_values(by='rules', ascending=False).reset_index()
 
    except ValueError as e:
        print(f'{e}')
        CP_detail_rules_techniques = None
        CP_agg_rules_techniques = None
    
    return CP_detail_rules_techniques, CP_agg_rules_techniques

In [899]:
def get_availability_rules_by_technique_and_tactics(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y táctica. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_tactics_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por táctica.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_tactics_NN_df,_,_,_,_,_,_,_,_ = get_techniques_tactics_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_tactics = pd.merge(CP_agg_rules_techniques, MITRE_techniques_tactics_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_tactics = CP_detail_rules_techniques_tactics[['tactic_ID', 'tactic', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('tactic_ID')
    CP_detail_rules_techniques_tactics = CP_detail_rules_techniques_tactics.reset_index(drop=True)
    CP_agg_rules_techniques_tactics = CP_detail_rules_techniques_tactics.groupby(['tactic_ID','tactic']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='tactic_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_tactics, CP_agg_rules_techniques_tactics

In [900]:
def get_availability_rules_by_technique_and_datasource(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y data source. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_datasources_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por data source.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_datasources_NN_df,_,_,_,_,_,_,_,_ = get_techniques_datasources_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_datasources = pd.merge(CP_agg_rules_techniques, MITRE_techniques_datasources_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_datasources = CP_detail_rules_techniques_datasources[['data_source_ID', 'data_source', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('data_source_ID')
    CP_detail_rules_techniques_datasources = CP_detail_rules_techniques_datasources.reset_index(drop=True)
    CP_agg_rules_techniques_datasources = CP_detail_rules_techniques_datasources.groupby(['data_source_ID','data_source']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='data_source_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_datasources, CP_agg_rules_techniques_datasources

In [901]:
def get_availability_rules_by_technique_and_platform(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y plataforma. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_platforms_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por plataforma.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_platforms_NN_df,_,_,_,_,_,_,_,_ = get_techniques_platforms_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_platforms = pd.merge(CP_agg_rules_techniques, MITRE_techniques_platforms_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_platforms = CP_detail_rules_techniques_platforms[['platform', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('platform')
    CP_detail_rules_techniques_platforms = CP_detail_rules_techniques_platforms.reset_index(drop=True)
    CP_agg_rules_techniques_platforms = CP_detail_rules_techniques_platforms.groupby(['platform']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='platform', ascending=True).reset_index()

    return CP_detail_rules_techniques_platforms, CP_agg_rules_techniques_platforms

In [902]:
def get_availability_rules_by_technique_and_group(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y grupo. Se llama a las funciones get_availability_rules_by_technique() y get_availability_rules_by_technique_and_group() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por grupo.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_groups_NN_df,_,_,_,_,_,_,_,_ = get_techniques_groups_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_groups = pd.merge(CP_agg_rules_techniques, MITRE_techniques_groups_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_groups = CP_detail_rules_techniques_groups[['group_ID', 'group', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('group_ID')
    CP_detail_rules_techniques_groups = CP_detail_rules_techniques_groups.reset_index(drop=True)
    CP_agg_rules_techniques_groups = CP_detail_rules_techniques_groups.groupby(['group_ID', 'group']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='group_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_groups, CP_agg_rules_techniques_groups

In [903]:
def get_availability_rules_by_technique_and_software(matrix_str, matrix_store, cp_techniques_list):
    '''
    Función encargada de obtener la disponibilidad de reglas por técnica y software. Se llama a las funciones get_availability_rules_by_technique() y get_techniques_software_relationships() de las cuales se recogen 2 dataframes para su posterior leftjoin. Se retornan 2 dataframes, por un lado un resumen detallado a nivel de táctica y técnica del número de reglas disponibles, y por otro lado un resumen agregado del numero de técnicas y reglas por software.
    '''
    _, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix_str)
    MITRE_techniques_software_NN_df,_,_,_,_,_,_,_,_ = get_techniques_software_relationships(matrix_store, cp_techniques_list)
    CP_detail_rules_techniques_software = pd.merge(CP_agg_rules_techniques, MITRE_techniques_software_NN_df, how='left', left_on='ttp', right_on='technique_ID')
    CP_detail_rules_techniques_software = CP_detail_rules_techniques_software[['software_ID', 'software', 'technique_ID', 'technique', 'rules', 'matrix']].sort_values('software_ID')
    CP_detail_rules_techniques_software = CP_detail_rules_techniques_software.reset_index(drop=True)
    CP_agg_rules_techniques_software = CP_detail_rules_techniques_software.groupby(['software_ID', 'software']).agg(
                                                                techniques=('technique', 'count'),
                                                                rules=('rules', 'sum'),
                                                                matrix=('matrix', 'first')
                                                            ).sort_values(by='software_ID', ascending=True).reset_index()

    return CP_detail_rules_techniques_software, CP_agg_rules_techniques_software

### **Parámetros**

In [904]:
matrix = 'ics' # enterprise / ics / mobile
save_as_csv = True
debug_df = True # Parámetro para controlar el printeado de df

# **Ejecución principal**

## **1. Elementos generales**

### **1.1 Generación de la matriz MITRE**

In [905]:
mitre_matrix = get_data_from_branch(matrix)
mitre_matrix

### **1.2 Generación de la lista de técnicas (ID) de la matriz MITRE seleccionada**

In [906]:
techniques_list = get_list_techniques_from_stix2(mitre_matrix,'both')
print(f"Se han generado la lista de técnicas (ID) para la matriz {matrix.upper()} que contiene un total de {len(techniques_list)} TTP's")

Se han generado la lista de técnicas (ID) para la matriz ICS que contiene un total de 83 TTP's


### **1.3 Obtención de las TTP disponbles en Cyber Proof con regla de detección**

In [907]:
cp_techniques =  get_CP_ttps_with_rules(matrix, way='file')
print(f"Se han obtenido un total de {len(cp_techniques)} TTP's con regla de detección asociada disponibles en CP.")

Se han obtenido un total de 32 TTP's con regla de detección asociada disponibles en CP.


## **2. Tablas informativas**

### **2.1. Técnicas**

#### **Generación de las tablas**

In [908]:
MITRE_techniques_df, CP_techniques_df, NOCP_techniques_df = get_techniques_information(mitre_matrix, cp_techniques, revoked_deprecated=True)

#### **Debug**

In [909]:
if debug_df:
    display(MITRE_techniques_df.head(3))
    display(CP_techniques_df.head(3))
    display(NOCP_techniques_df.head(3))
    print(f'{MITRE_techniques_df.shape}, {CP_techniques_df.shape}, {NOCP_techniques_df.shape}')

,technique_ID,technique,technique_url,technique_description,technique_deprecated,technique_revoked,matrix_domains
0,T0803,Block Command Message,https://attack.mitre.org/techniques/T0803,Adversaries may block a command message from r...,False,False,[ics-attack]
1,T0881,Service Stop,https://attack.mitre.org/techniques/T0881,Adversaries may stop or disable services on a ...,False,False,[ics-attack]
2,T0836,Modify Parameter,https://attack.mitre.org/techniques/T0836,Adversaries may modify parameters used to inst...,False,False,[ics-attack]


,technique_ID,technique,technique_url,technique_description,technique_deprecated,technique_revoked,matrix_domains
2,T0836,Modify Parameter,https://attack.mitre.org/techniques/T0836,Adversaries may modify parameters used to inst...,False,False,[ics-attack]
5,T0829,Loss of View,https://attack.mitre.org/techniques/T0829,Adversaries may cause a sustained or permanent...,False,False,[ics-attack]
7,T0831,Manipulation of Control,https://attack.mitre.org/techniques/T0831,Adversaries may manipulate physical process co...,False,False,[ics-attack]


,technique_ID,technique,technique_url,technique_description,technique_deprecated,technique_revoked,matrix_domains
0,T0803,Block Command Message,https://attack.mitre.org/techniques/T0803,Adversaries may block a command message from r...,False,False,[ics-attack]
1,T0881,Service Stop,https://attack.mitre.org/techniques/T0881,Adversaries may stop or disable services on a ...,False,False,[ics-attack]
3,T0821,Modify Controller Tasking,https://attack.mitre.org/techniques/T0821,Adversaries may modify the tasking of a contro...,False,False,[ics-attack]


(83, 7), (32, 7), (51, 7)


#### **Guardado**

In [910]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_df, f'[MITRE]_{matrix}_techniques',  matrix, 'information/techniques')
    save_df_as_csv(CP_techniques_df, f'[CP]_{matrix}_techniques', matrix, 'information/techniques')
    save_df_as_csv(NOCP_techniques_df, f'[NOCP]_{matrix}_techniques', matrix, 'information/techniques')

Archivo guardado correctamente [MITRE]_ics_techniques.csv
Archivo guardado correctamente [CP]_ics_techniques.csv
Archivo guardado correctamente [NOCP]_ics_techniques.csv


### **2.2. Tácticas**

#### **Generación de la tabla**

In [911]:
MITRE_tactics_df = get_tactics_information(mitre_matrix)

#### **Debug**

In [912]:
if debug_df:
    display(MITRE_tactics_df.head(3))
    print(f'{MITRE_tactics_df.shape}')

,tactic_ID,tactic,tactic_url,tactic_description,matrix_domains
0,TA0107,Inhibit Response Function,https://attack.mitre.org/tactics/TA0107,The adversary is trying to prevent your safety...,[ics-attack]
1,TA0111,Privilege Escalation,https://attack.mitre.org/tactics/TA0111,The adversary is trying to gain higher-level p...,[ics-attack]
2,TA0109,Lateral Movement,https://attack.mitre.org/tactics/TA0109,The adversary is trying to move through your I...,[ics-attack]


(12, 5)


#### **Guardado**

In [913]:
if save_as_csv:
    save_df_as_csv(MITRE_tactics_df, f'[MITRE]_{matrix}_tactics', matrix, 'information/tactics')

Archivo guardado correctamente [MITRE]_ics_tactics.csv


### **2.3. Data sources**

#### **Generación de las tablas**

In [914]:
MITRE_datasources_df = get_datasources_information(mitre_matrix, False)

#### **Debug**

In [915]:
if debug_df:
    display(MITRE_datasources_df.head(3))
    print(f'{MITRE_datasources_df.shape}')

,data_source_ID,data_source,data_source_url,data_source_description,data_source_deprecated,data_source_revoked,matrix_domains
0,DS0015,Application Log,https://attack.mitre.org/datasources/DS0015,Events collected by third-party services such ...,False,False,"[enterprise-attack, ics-attack]"
1,DS0002,User Account,https://attack.mitre.org/datasources/DS0002,"A profile representing a user, device, service...",False,False,[enterprise-attack]
2,DS0033,Network Share,https://attack.mitre.org/datasources/DS0033,A storage resource (typically a folder or driv...,False,False,[enterprise-attack]


(17, 7)


#### **Guardado**

In [916]:
if save_as_csv:
    save_df_as_csv(MITRE_datasources_df, f'[MITRE]_{matrix}_datasources', matrix, 'information/datasources')

Archivo guardado correctamente [MITRE]_ics_datasources.csv


### **2.4. Plataformas**

#### **Generación de las tablas**

In [917]:
MITRE_platforms_df = get_platforms_information(mitre_matrix)

#### **Debug**

In [918]:
if debug_df:
    display(MITRE_platforms_df.head(3))
    print(f'{MITRE_platforms_df.shape}')

,platform
0,None


(1, 1)


#### **Guardado**

In [919]:
if save_as_csv:
    save_df_as_csv(MITRE_platforms_df, f'[MITRE]_{matrix}_platforms', matrix, 'information/platforms')


Archivo guardado correctamente [MITRE]_ics_platforms.csv


### **2.5. Grupos**

#### **Generación de las tablas**

In [920]:
MITRE_groups_df = get_groups_information(mitre_matrix, revoked_deprecated=True)

#### **Debug**

In [921]:
if debug_df:
    display(MITRE_groups_df.head(3))
    print(f'{MITRE_groups_df.shape}')

,group_ID,group,group_aliases,group_url,group_description,group_deprecated,group_revoked,matrix_domains
0,G0032,Lazarus Group,"[Lazarus Group, Labyrinth Chollima, HIDDEN COB...",https://attack.mitre.org/groups/G0032,[Lazarus Group](https://attack.mitre.org/group...,False,False,"[enterprise-attack, ics-attack]"
1,G0034,Sandworm Team,"[Sandworm Team, ELECTRUM, Telebots, IRON VIKIN...",https://attack.mitre.org/groups/G0034,[Sandworm Team](https://attack.mitre.org/group...,False,False,"[enterprise-attack, ics-attack, mobile-attack]"
2,G0035,Dragonfly,"[Dragonfly, TEMP.Isotope, DYMALLOY, Berserk Be...",https://attack.mitre.org/groups/G0035,[Dragonfly](https://attack.mitre.org/groups/G0...,False,False,"[enterprise-attack, ics-attack]"


(14, 8)


#### **Guardado**

In [922]:
if save_as_csv:
    save_df_as_csv(MITRE_groups_df, f'[MITRE]_{matrix}_groups', matrix, 'information/groups')

Archivo guardado correctamente [MITRE]_ics_groups.csv


### **2.6. Software**

#### **Generación de las tablas**

In [923]:
MITRE_software_df = get_software_information(mitre_matrix, revoked_deprecated=True)

#### **Debug**

In [924]:
if debug_df:
    display(MITRE_software_df.head(3))
    print(f'{MITRE_software_df.shape}')

,software_ID,software,software_type,software_url,software_description,software_deprecated,software_revoked,matrix_domains
0,S0038,Duqu,malware,https://attack.mitre.org/software/S0038,[Duqu](https://attack.mitre.org/software/S0038...,False,False,"[enterprise-attack, ics-attack]"
1,S0089,BlackEnergy,malware,https://attack.mitre.org/software/S0089,[BlackEnergy](https://attack.mitre.org/softwar...,False,False,"[enterprise-attack, ics-attack]"
2,S0093,Backdoor.Oldrea,malware,https://attack.mitre.org/software/S0093,[Backdoor.Oldrea](https://attack.mitre.org/sof...,False,False,"[enterprise-attack, ics-attack]"


(21, 8)


#### **Guardado**

In [925]:
if save_as_csv:
    save_df_as_csv(MITRE_software_df, f'[MITRE]_{matrix}_software', matrix, 'information/software')

Archivo guardado correctamente [MITRE]_ics_software.csv


## **3. Relaciones**

### **3.1. Relación técnicas - tácticas**

#### **Generación de las tablas**

In [926]:
MITRE_techniques_tactics_NN_df, CP_techniques_tactics_NN_df, NOCP_techniques_tactics_NN_df, MITRE_technique_tactics_1N_df, CP_technique_tactics_1N_df, NOCP_technique_tactics_1N_df, MITRE_techniques_tactic_N1_df, CP_techniques_tactic_N1_df, NOCP_techniques_tactic_N1_df = get_techniques_tactics_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [927]:
if debug_df:
    display(MITRE_techniques_tactics_NN_df.head(3))
    display(CP_techniques_tactics_NN_df.head(3))
    display(NOCP_techniques_tactics_NN_df.head(3))
    print(f'{MITRE_techniques_tactics_NN_df.shape}, {CP_techniques_tactics_NN_df.shape}, {NOCP_techniques_tactics_NN_df.shape}')

,technique_ID,technique,tactic_ID,tactic
0,T0803,Block Command Message,TA0107,Inhibit Response Function
1,T0881,Service Stop,TA0107,Inhibit Response Function
2,T0836,Modify Parameter,TA0106,Impair Process Control


,technique_ID,technique,tactic_ID,tactic
2,T0836,Modify Parameter,TA0106,Impair Process Control
6,T0829,Loss of View,TA0105,Impact
8,T0831,Manipulation of Control,TA0105,Impact


,technique_ID,technique,tactic_ID,tactic
0,T0803,Block Command Message,TA0107,Inhibit Response Function
1,T0881,Service Stop,TA0107,Inhibit Response Function
3,T0821,Modify Controller Tasking,TA0104,Execution


(94, 4), (37, 4), (57, 4)


In [928]:
if debug_df:
    display(MITRE_technique_tactics_1N_df.head(3))
    display(CP_technique_tactics_1N_df.head(3))
    display(NOCP_technique_tactics_1N_df.head(3))
    print(f'{MITRE_technique_tactics_1N_df.shape}, {CP_technique_tactics_1N_df.shape}, {NOCP_technique_tactics_1N_df.shape}')

,technique_ID,technique,tactic_ID,tactic
0,T0800,[Activate Firmware Update Mode],[TA0107],[Inhibit Response Function]
1,T0801,[Monitor Process State],[TA0100],[Collection]
2,T0802,[Automated Collection],[TA0100],[Collection]


,technique_ID,technique,tactic_ID,tactic
0,T0802,[Automated Collection],[TA0100],[Collection]
1,T0807,[Command-Line Interface],[TA0104],[Execution]
2,T0813,[Denial of Control],[TA0105],[Impact]


,technique_ID,technique,tactic_ID,tactic
0,T0800,[Activate Firmware Update Mode],[TA0107],[Inhibit Response Function]
1,T0801,[Monitor Process State],[TA0100],[Collection]
2,T0803,[Block Command Message],[TA0107],[Inhibit Response Function]


(83, 4), (32, 4), (51, 4)


In [929]:
if debug_df:
    display(MITRE_techniques_tactic_N1_df.head(3))
    display(CP_techniques_tactic_N1_df.head(3))
    display(NOCP_techniques_tactic_N1_df.head(3))
    print(f'{MITRE_techniques_tactic_N1_df.shape}, {CP_techniques_tactic_N1_df.shape}, {NOCP_techniques_tactic_N1_df.shape}')

,tactic_ID,tactic,technique_ID,technique
0,TA0100,[Collection],"[T0845, T0861, T0811, T0877, T0852, T0893, T08...","[Detect Operating Mode, Monitor Process State,..."
1,TA0101,[Command and Control],"[T0869, T0884, T0885]","[Connection Proxy, Standard Application Layer ..."
2,TA0102,[Discovery],"[T0888, T0840, T0846, T0887, T0842]","[Network Sniffing, Remote System Discovery, Wi..."


,tactic_ID,tactic,technique_ID,technique
0,TA0100,[Collection],"[T0802, T0868]","[Automated Collection, Detect Operating Mode]"
1,TA0101,[Command and Control],"[T0869, T0884, T0885]","[Connection Proxy, Standard Application Layer ..."
2,TA0102,[Discovery],"[T0840, T0842]","[Network Sniffing, Network Connection Enumerat..."


,tactic_ID,tactic,technique_ID,technique
0,TA0100,[Collection],"[T0845, T0861, T0811, T0877, T0852, T0893, T08...","[Monitor Process State, Wireless Sniffing, Poi..."
1,TA0102,[Discovery],"[T0887, T0846, T0888]","[Wireless Sniffing, Remote System Discovery, R..."
2,TA0103,[Evasion],"[T0851, T0894, T0872, T0820, T0849]","[Rootkit, Exploitation for Evasion, System Bin..."


(12, 4), (12, 4), (11, 4)


#### **Guardado**

In [930]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_tactics_NN_df, f'[MITRE]_{matrix}_techniques_tactics_NN', matrix, 'relations/techniques_tactics')
    save_df_as_csv(CP_techniques_tactics_NN_df, f'[CP]_{matrix}_techniques_tactics_NN', matrix, 'relations/techniques_tactics')
    save_df_as_csv(NOCP_techniques_tactics_NN_df, f'[NOCP]_{matrix}_techniques_tactics_NN', matrix, 'relations/techniques_tactics')

    save_df_as_csv(MITRE_technique_tactics_1N_df, f'[MITRE]_{matrix}_technique_tactics_1N', matrix, 'relations/techniques_tactics')
    save_df_as_csv(CP_technique_tactics_1N_df, f'[CP]_{matrix}_technique_tactics_1N', matrix, 'relations/techniques_tactics')
    save_df_as_csv(NOCP_technique_tactics_1N_df, f'[NOCP]_{matrix}_technique_tactics_1N', matrix, 'relations/techniques_tactics')

    save_df_as_csv(MITRE_techniques_tactic_N1_df, f'[MITRE]_{matrix}_techniques_tactic_N1', matrix, 'relations/techniques_tactics')
    save_df_as_csv(CP_techniques_tactic_N1_df, f'[CP]_{matrix}_techniques_tactic_N1', matrix, 'relations/techniques_tactics')
    save_df_as_csv(NOCP_techniques_tactic_N1_df, f'[NOCP]_{matrix}_techniques_tactic_N1', matrix, 'relations/techniques_tactics')

Archivo guardado correctamente [MITRE]_ics_techniques_tactics_NN.csv
Archivo guardado correctamente [CP]_ics_techniques_tactics_NN.csv
Archivo guardado correctamente [NOCP]_ics_techniques_tactics_NN.csv
Archivo guardado correctamente [MITRE]_ics_technique_tactics_1N.csv
Archivo guardado correctamente [CP]_ics_technique_tactics_1N.csv
Archivo guardado correctamente [NOCP]_ics_technique_tactics_1N.csv
Archivo guardado correctamente [MITRE]_ics_techniques_tactic_N1.csv
Archivo guardado correctamente [CP]_ics_techniques_tactic_N1.csv
Archivo guardado correctamente [NOCP]_ics_techniques_tactic_N1.csv


### **3.2. Relación técnicas - data sources**

#### **Generación de las tablas**

In [931]:
MITRE_techniques_datasources_NN_df, CP_techniques_datasources_NN_df, NOCP_techniques_datasources_NN_df, MITRE_technique_datasources_1N_df, CP_technique_datasources_1N_df, NOCP_technique_datasources_1N_df, MITRE_techniques_datasource_N1_df, CP_techniques_datasource_N1_df, NOCP_techniques_datasource_N1_df = get_techniques_datasources_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [932]:
if debug_df:
    display(MITRE_techniques_datasources_NN_df.head(3))
    display(CP_techniques_datasources_NN_df.head(3))
    display(NOCP_techniques_datasources_NN_df.head(3))
    print(f'{MITRE_techniques_datasources_NN_df.shape}, {CP_techniques_datasources_NN_df.shape}, {NOCP_techniques_datasources_NN_df.shape}')

,technique_ID,technique,data_source_ID,data_source
0,T0800,Activate Firmware Update Mode,DS0015,Application Log
1,T0800,Activate Firmware Update Mode,DS0040,Operational Databases
2,T0800,Activate Firmware Update Mode,DS0029,Network Traffic


,technique_ID,technique,data_source_ID,data_source
5,T0802,Automated Collection,DS0012,Script
6,T0802,Automated Collection,DS0017,Command
7,T0802,Automated Collection,DS0022,File


,technique_ID,technique,data_source_ID,data_source
0,T0800,Activate Firmware Update Mode,DS0015,Application Log
1,T0800,Activate Firmware Update Mode,DS0040,Operational Databases
2,T0800,Activate Firmware Update Mode,DS0029,Network Traffic


(210, 4), (74, 4), (136, 4)


In [933]:
if debug_df:
    display(MITRE_technique_datasources_1N_df.head(3))
    display(CP_technique_datasources_1N_df.head(3))
    display(NOCP_technique_datasources_1N_df.head(3))
    print(f'{MITRE_technique_datasources_1N_df.shape}, {CP_technique_datasources_1N_df.shape}, {NOCP_technique_datasources_1N_df.shape}')

,technique_ID,technique,data_source_ID,data_source
0,T0800,[Activate Firmware Update Mode],"[DS0040, DS0029, DS0015]","[Network Traffic, Operational Databases, Appli..."
1,T0801,[Monitor Process State],"[DS0029, DS0015]","[Network Traffic, Application Log]"
2,T0802,[Automated Collection],"[DS0029, DS0012, DS0022, DS0017]","[Command, Network Traffic, Script, File]"


,technique_ID,technique,data_source_ID,data_source
0,T0802,[Automated Collection],"[DS0029, DS0012, DS0022, DS0017]","[Command, Network Traffic, Script, File]"
1,T0807,[Command-Line Interface],"[DS0009, DS0015, DS0017]","[Command, Process, Application Log]"
2,T0813,[Denial of Control],[nan],[]


,technique_ID,technique,data_source_ID,data_source
0,T0800,[Activate Firmware Update Mode],"[DS0040, DS0029, DS0015]","[Network Traffic, Operational Databases, Appli..."
1,T0801,[Monitor Process State],"[DS0029, DS0015]","[Network Traffic, Application Log]"
2,T0803,[Block Command Message],"[DS0029, DS0009, DS0040, DS0015]","[Process, Network Traffic, Operational Databas..."


(83, 4), (32, 4), (51, 4)


In [934]:
if debug_df:
    display(MITRE_techniques_datasource_N1_df.head(3))
    display(CP_techniques_datasource_N1_df.head(3))
    display(NOCP_techniques_datasource_N1_df.head(3))
    print(f'{MITRE_techniques_datasource_N1_df.shape}, {CP_techniques_datasource_N1_df.shape}, {NOCP_techniques_datasource_N1_df.shape}')

,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T0857, T0851, T0839]","[System Firmware, Module Firmware, Rootkit]"
1,DS0002,[User Account],[T0859],[Valid Accounts]
2,DS0003,[Scheduled Job],[T0849],[Masquerading]


,data_source_ID,data_source,technique_ID,technique
0,DS0002,[User Account],[T0859],[Valid Accounts]
1,DS0009,[Process],"[T0863, T0853, T0840, T0867, T0807, T0842, T0886]","[Network Sniffing, Command-Line Interface, Use..."
2,DS0011,[Module],"[T0886, T0853]","[Remote Services, Scripting]"


,data_source_ID,data_source,technique_ID,technique
0,DS0001,[Firmware],"[T0857, T0851, T0839]","[System Firmware, Module Firmware, Rootkit]"
1,DS0003,[Scheduled Job],[T0849],[Masquerading]
2,DS0009,[Process],"[T0809, T0805, T0852, T0803, T0823, T0847, T08...","[Block Serial COM, Drive-by Compromise, Autoru..."


(17, 4), (13, 4), (16, 4)


#### **Guardado**

In [935]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_datasources_NN_df, f'[MITRE]_{matrix}_techniques_datasources_NN', matrix, 'relations/techniques_datasources')
    save_df_as_csv(CP_techniques_datasources_NN_df, f'[CP]_{matrix}_techniques_datasources_NN', matrix, 'relations/techniques_datasources')
    save_df_as_csv(NOCP_techniques_datasources_NN_df, f'[NOCP]_{matrix}_techniques_datasources_NN', matrix, 'relations/techniques_datasources')

    save_df_as_csv(MITRE_technique_datasources_1N_df, f'[MITRE]_{matrix}_technique_datasources_1N', matrix, 'relations/techniques_datasources')
    save_df_as_csv(CP_technique_datasources_1N_df, f'[CP]_{matrix}_technique_datasources_1N', matrix, 'relations/techniques_datasources')
    save_df_as_csv(NOCP_technique_datasources_1N_df, f'[NOCP]_{matrix}_technique_datasources_1N', matrix, 'relations/techniques_datasources')

    save_df_as_csv(MITRE_techniques_datasource_N1_df, f'[MITRE]_{matrix}_techniques_datasource_N1', matrix, 'relations/techniques_datasources')
    save_df_as_csv(CP_techniques_datasource_N1_df, f'[CP]_{matrix}_techniques_datasource_N1', matrix, 'relations/techniques_datasources')
    save_df_as_csv(NOCP_techniques_datasource_N1_df, f'[NOCP]_{matrix}_techniques_datasource_N1', matrix, 'relations/techniques_datasources')

Archivo guardado correctamente [MITRE]_ics_techniques_datasources_NN.csv
Archivo guardado correctamente [CP]_ics_techniques_datasources_NN.csv
Archivo guardado correctamente [NOCP]_ics_techniques_datasources_NN.csv
Archivo guardado correctamente [MITRE]_ics_technique_datasources_1N.csv
Archivo guardado correctamente [CP]_ics_technique_datasources_1N.csv
Archivo guardado correctamente [NOCP]_ics_technique_datasources_1N.csv
Archivo guardado correctamente [MITRE]_ics_techniques_datasource_N1.csv
Archivo guardado correctamente [CP]_ics_techniques_datasource_N1.csv
Archivo guardado correctamente [NOCP]_ics_techniques_datasource_N1.csv


### **3.3. Relación técnicas - plataformas**

#### **Generación de las tablas**

In [936]:
MITRE_techniques_platforms_NN_df, CP_techniques_platforms_NN_df, NOCP_techniques_platforms_NN_df, MITRE_technique_platforms_1N_df, CP_technique_platforms_1N_df, NOCP_technique_platforms_1N_df, MITRE_techniques_platform_N1_df, CP_techniques_platform_N1_df, NOCP_techniques_platform_N1_df = get_techniques_platforms_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [937]:
if debug_df:
    display(MITRE_techniques_platforms_NN_df.head(3))
    display(CP_techniques_platforms_NN_df.head(3))
    display(NOCP_techniques_platforms_NN_df.head(3))
    print(f'{MITRE_techniques_platforms_NN_df.shape}, {CP_techniques_platforms_NN_df.shape}, {NOCP_techniques_platforms_NN_df.shape}')

,technique_ID,technique,platform
0,T0800,Activate Firmware Update Mode,None
1,T0801,Monitor Process State,None
2,T0802,Automated Collection,None


,technique_ID,technique,platform
2,T0802,Automated Collection,None
7,T0807,Command-Line Interface,None
11,T0813,Denial of Control,None


,technique_ID,technique,platform
0,T0800,Activate Firmware Update Mode,None
1,T0801,Monitor Process State,None
3,T0803,Block Command Message,None


(83, 3), (32, 3), (51, 3)


In [938]:
if debug_df:
    display(MITRE_technique_platforms_1N_df.head(3))
    display(CP_technique_platforms_1N_df.head(3))
    display(NOCP_technique_platforms_1N_df.head(3))
    print(f'{MITRE_technique_platforms_1N_df.shape}, {CP_technique_platforms_1N_df.shape}, {NOCP_technique_platforms_1N_df.shape}')

,technique_ID,technique,platform
0,T0800,[Activate Firmware Update Mode],[None]
1,T0801,[Monitor Process State],[None]
2,T0802,[Automated Collection],[None]


,technique_ID,technique,platform
0,T0802,[Automated Collection],[None]
1,T0807,[Command-Line Interface],[None]
2,T0813,[Denial of Control],[None]


,technique_ID,technique,platform
0,T0800,[Activate Firmware Update Mode],[None]
1,T0801,[Monitor Process State],[None]
2,T0803,[Block Command Message],[None]


(83, 3), (32, 3), (51, 3)


In [939]:
if debug_df:
    display(MITRE_techniques_platform_N1_df.head(3))
    display(CP_techniques_platform_N1_df.head(3))
    display(NOCP_techniques_platform_N1_df.head(3))
    print(f'{MITRE_techniques_platform_N1_df.shape}, {CP_techniques_platform_N1_df.shape}, {NOCP_techniques_platform_N1_df.shape}')

,platform,technique_ID,technique
0,None,"[T0809, T0861, T0805, T0811, T0822, T0889, T08...","[Block Serial COM, Exploitation for Evasion, A..."


,platform,technique_ID,technique
0,None,"[T0885, T0828, T0858, T0822, T0866, T0829, T08...","[Damage to Property, Commonly Used Port, Loss ..."


,platform,technique_ID,technique
0,None,"[T0809, T0861, T0805, T0811, T0848, T0803, T08...","[Brute Force I/O, Block Serial COM, Rootkit, E..."


(1, 3), (1, 3), (1, 3)


#### **Guardado**

In [940]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_platforms_NN_df, f'[MITRE]_{matrix}_techniques_platforms_NN', matrix, 'relations/techniques_platforms')
    save_df_as_csv(CP_techniques_platforms_NN_df, f'[CP]_{matrix}_techniques_platforms_NN', matrix, 'relations/techniques_platforms')
    save_df_as_csv(NOCP_techniques_platforms_NN_df, f'[NOCP]_{matrix}_techniques_platforms_NN', matrix, 'relations/techniques_platforms')

    save_df_as_csv(MITRE_technique_platforms_1N_df, f'[MITRE]_{matrix}_technique_platforms_1N', matrix, 'relations/techniques_platforms')
    save_df_as_csv(CP_technique_platforms_1N_df, f'[CP]_{matrix}_technique_platforms_1N', matrix, 'relations/techniques_platforms')
    save_df_as_csv(NOCP_technique_platforms_1N_df, f'[NOCP]_{matrix}_technique_platforms_1N', matrix, 'relations/techniques_platforms')

    save_df_as_csv(MITRE_techniques_platform_N1_df, f'[MITRE]_{matrix}_techniques_platform_N1', matrix, 'relations/techniques_platforms')
    save_df_as_csv(CP_techniques_platform_N1_df, f'[CP]_{matrix}_techniques_platform_N1', matrix, 'relations/techniques_platforms')
    save_df_as_csv(NOCP_techniques_platform_N1_df, f'[NOCP]_{matrix}_techniques_platform_N1', matrix, 'relations/techniques_platforms')

Archivo guardado correctamente [MITRE]_ics_techniques_platforms_NN.csv
Archivo guardado correctamente [CP]_ics_techniques_platforms_NN.csv
Archivo guardado correctamente [NOCP]_ics_techniques_platforms_NN.csv
Archivo guardado correctamente [MITRE]_ics_technique_platforms_1N.csv
Archivo guardado correctamente [CP]_ics_technique_platforms_1N.csv
Archivo guardado correctamente [NOCP]_ics_technique_platforms_1N.csv
Archivo guardado correctamente [MITRE]_ics_techniques_platform_N1.csv
Archivo guardado correctamente [CP]_ics_techniques_platform_N1.csv
Archivo guardado correctamente [NOCP]_ics_techniques_platform_N1.csv


### **3.4. Relación técnicas - grupos**

#### **Generación de las tablas**

In [941]:
MITRE_techniques_groups_NN_df, CP_techniques_groups_NN_df, NOCP_techniques_groups_NN_df, MITRE_technique_groups_1N_df, CP_technique_groups_1N_df, NOCP_technique_groups_1N_df, MITRE_techniques_group_N1_df, CP_techniques_group_N1_df, NOCP_techniques_group_N1_df = get_techniques_groups_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [942]:
if debug_df:
    display(MITRE_techniques_groups_NN_df.head(3))
    display(CP_techniques_groups_NN_df.head(3))
    display(NOCP_techniques_groups_NN_df.head(3))
    print(f'{MITRE_techniques_groups_NN_df.shape}, {CP_techniques_groups_NN_df.shape}, {NOCP_techniques_platforms_NN_df.shape}')

,group_ID,group,technique_ID,technique
0,G0034,Sandworm Team,T0807,Command-Line Interface
1,G0035,Dragonfly,T0817,Drive-by Compromise
2,G0088,TEMP.Veles,T0817,Drive-by Compromise


,group_ID,group,technique_ID,technique
0,G0034,Sandworm Team,T0807,Command-Line Interface
8,G0049,OilRig,T0853,Scripting
9,G0064,APT33,T0853,Scripting


,group_ID,group,technique_ID,technique
1,G0035,Dragonfly,T0817,Drive-by Compromise
2,G0088,TEMP.Veles,T0817,Drive-by Compromise
3,G0049,OilRig,T0817,Drive-by Compromise


(20, 4), (7, 4), (51, 3)


In [943]:
if debug_df:
    display(MITRE_technique_groups_1N_df.head(3))
    display(CP_technique_groups_1N_df.head(3))
    display(NOCP_technique_groups_1N_df.head(3))
    print(f'{MITRE_technique_groups_1N_df.shape}, {CP_technique_groups_1N_df.shape}, {NOCP_technique_groups_1N_df.shape}')

,technique_ID,technique,group_ID,group
0,T0807,[Command-Line Interface],[G0034],[Sandworm Team]
1,T0817,[Drive-by Compromise],"[G0049, G0035, G0088, G1000]","[OilRig, Dragonfly, ALLANITE, TEMP.Veles]"
2,T0819,[Exploit Public-Facing Application],[G0034],[Sandworm Team]


,technique_ID,technique,group_ID,group
0,T0807,[Command-Line Interface],[G0034],[Sandworm Team]
1,T0853,[Scripting],"[G0064, G0049]","[APT33, OilRig]"
2,T0859,[Valid Accounts],"[G1000, G0049]","[OilRig, ALLANITE]"


,technique_ID,technique,group_ID,group
0,T0817,[Drive-by Compromise],"[G0049, G0035, G0088, G1000]","[OilRig, Dragonfly, ALLANITE, TEMP.Veles]"
1,T0819,[Exploit Public-Facing Application],[G0034],[Sandworm Team]
2,T0852,[Screen Capture],"[G0064, G1000]","[APT33, ALLANITE]"


(10, 4), (5, 4), (5, 4)


In [944]:
if debug_df:
    display(MITRE_techniques_group_N1_df.head(3))
    display(CP_techniques_group_N1_df.head(3))
    display(NOCP_techniques_group_N1_df.head(3))
    print(f'{MITRE_techniques_group_N1_df.shape}, {CP_techniques_group_N1_df.shape}, {NOCP_techniques_group_N1_df.shape}')

,group_ID,group,technique_ID,technique
0,G0032,[Lazarus Group],[T0865],[Spearphishing Attachment]
1,G0034,[Sandworm Team],"[T0807, T0819, T0884]","[Connection Proxy, Command-Line Interface, Exp..."
2,G0035,[Dragonfly],"[T0817, T0862]","[Supply Chain Compromise, Drive-by Compromise]"


,group_ID,group,technique_ID,technique
0,G0034,[Sandworm Team],"[T0807, T0884]","[Connection Proxy, Command-Line Interface]"
1,G0049,[OilRig],"[T0869, T0859, T0853]","[Valid Accounts, Standard Application Layer Pr..."
2,G0064,[APT33],[T0853],[Scripting]


,group_ID,group,technique_ID,technique
0,G0032,[Lazarus Group],[T0865],[Spearphishing Attachment]
1,G0034,[Sandworm Team],[T0819],[Exploit Public-Facing Application]
2,G0035,[Dragonfly],"[T0817, T0862]","[Supply Chain Compromise, Drive-by Compromise]"


(7, 4), (4, 4), (7, 4)


#### **Guardado**

In [945]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_groups_NN_df, f'[MITRE]_{matrix}_techniques_groups_NN', matrix, 'relations/techniques_groups')
    save_df_as_csv(CP_techniques_groups_NN_df, f'[CP]_{matrix}_techniques_groups_NN', matrix, 'relations/techniques_groups')
    save_df_as_csv(NOCP_techniques_groups_NN_df, f'[NOCP]_{matrix}_techniques_groups_NN', matrix, 'relations/techniques_groups')

    save_df_as_csv(MITRE_technique_groups_1N_df, f'[MITRE]_{matrix}_technique_groups_1N', matrix, 'relations/techniques_groups')
    save_df_as_csv(CP_technique_groups_1N_df, f'[CP]_{matrix}_technique_groups_1N', matrix, 'relations/techniques_groups')
    save_df_as_csv(NOCP_technique_groups_1N_df, f'[NOCP]_{matrix}_technique_groups_1N', matrix, 'relations/techniques_groups')

    save_df_as_csv(MITRE_techniques_group_N1_df, f'[MITRE]_{matrix}_techniques_group_N1', matrix, 'relations/techniques_groups')
    save_df_as_csv(CP_techniques_group_N1_df, f'[CP]_{matrix}_techniques_group_N1', matrix, 'relations/techniques_groups')
    save_df_as_csv(NOCP_techniques_group_N1_df, f'[NOCP]_{matrix}_techniques_group_N1', matrix, 'relations/techniques_groups')

Archivo guardado correctamente [MITRE]_ics_techniques_groups_NN.csv
Archivo guardado correctamente [CP]_ics_techniques_groups_NN.csv
Archivo guardado correctamente [NOCP]_ics_techniques_groups_NN.csv
Archivo guardado correctamente [MITRE]_ics_technique_groups_1N.csv
Archivo guardado correctamente [CP]_ics_technique_groups_1N.csv
Archivo guardado correctamente [NOCP]_ics_technique_groups_1N.csv
Archivo guardado correctamente [MITRE]_ics_techniques_group_N1.csv
Archivo guardado correctamente [CP]_ics_techniques_group_N1.csv
Archivo guardado correctamente [NOCP]_ics_techniques_group_N1.csv


### **3.5. Relación técnicas - software**

#### **Generación de las tablas**

In [946]:
MITRE_techniques_software_NN_df, CP_techniques_software_NN_df, NOCP_techniques_software_NN_df, MITRE_technique_software_1N_df, CP_technique_software_1N_df, NOCP_technique_software_1N_df, MITRE_techniques_software_N1_df, CP_techniques_software_N1_df, NOCP_techniques_software_N1_df = get_techniques_software_relationships(mitre_matrix, cp_techniques)

Dataframes generados correctamente!


#### **Debug**

In [947]:
if debug_df:
    display(MITRE_techniques_software_NN_df.head(3))
    display(CP_techniques_software_NN_df.head(3))
    display(NOCP_techniques_software_NN_df.head(3))
    print(f'{MITRE_techniques_software_NN_df.shape}, {CP_techniques_software_NN_df.shape}, {NOCP_techniques_software_NN_df.shape}')

,technique_ID,technique,software_ID,software
0,T0800,Activate Firmware Update Mode,S0604,Industroyer
1,T0801,Monitor Process State,S1072,Industroyer2
2,T0801,Monitor Process State,S0603,Stuxnet


,technique_ID,technique,software_ID,software
4,T0802,Automated Collection,S1072,Industroyer2
5,T0802,Automated Collection,S0093,Backdoor.Oldrea
6,T0802,Automated Collection,S0604,Industroyer


,technique_ID,technique,software_ID,software
0,T0800,Activate Firmware Update Mode,S0604,Industroyer
1,T0801,Monitor Process State,S1072,Industroyer2
2,T0801,Monitor Process State,S0603,Stuxnet


(152, 4), (76, 4), (76, 4)


In [948]:
if debug_df:
    display(MITRE_technique_software_1N_df.head(3))
    display(CP_technique_software_1N_df.head(3))
    display(NOCP_technique_software_1N_df.head(3))
    print(f'{MITRE_technique_software_1N_df.shape}, {CP_technique_software_1N_df.shape}, {NOCP_technique_software_1N_df.shape}')

,technique_ID,technique,software_ID,software
0,T0800,[Activate Firmware Update Mode],[S0604],[Industroyer]
1,T0801,[Monitor Process State],"[S1072, S0604, S0603]","[Industroyer2, Stuxnet, Industroyer]"
2,T0802,[Automated Collection],"[S0093, S1072, S0604]","[Industroyer2, Backdoor.Oldrea, Industroyer]"


,technique_ID,technique,software_ID,software
0,T0802,[Automated Collection],"[S0093, S1072, S0604]","[Industroyer2, Backdoor.Oldrea, Industroyer]"
1,T0807,[Command-Line Interface],"[S0604, S0603]","[Stuxnet, Industroyer]"
2,T0813,[Denial of Control],[S0604],[Industroyer]


,technique_ID,technique,software_ID,software
0,T0800,[Activate Firmware Update Mode],[S0604],[Industroyer]
1,T0801,[Monitor Process State],"[S1072, S0604, S0603]","[Industroyer2, Stuxnet, Industroyer]"
2,T0803,[Block Command Message],[S0604],[Industroyer]


(65, 4), (28, 4), (37, 4)


In [949]:
if debug_df:
    display(MITRE_techniques_software_N1_df.head(3))
    display(CP_techniques_software_N1_df.head(3))
    display(NOCP_techniques_software_N1_df.head(3))
    print(f'{MITRE_techniques_software_N1_df.shape}, {CP_techniques_software_N1_df.shape}, {NOCP_techniques_software_N1_df.shape}')

,software_ID,software,technique_ID,technique
0,S0038,[Duqu],"[T0882, T0811, T0893]","[Theft of Operational Information, Data from L..."
1,S0089,[BlackEnergy],"[T0869, T0859, T0865]","[Valid Accounts, Standard Application Layer Pr..."
2,S0093,[Backdoor.Oldrea],"[T0861, T0863, T0888, T0865, T0814, T0846, T08...","[Remote System Information Discovery, Remote S..."


,software_ID,software,technique_ID,technique
0,S0038,[Duqu],[T0882],[Theft of Operational Information]
1,S0089,[BlackEnergy],"[T0869, T0859]","[Valid Accounts, Standard Application Layer Pr..."
2,S0093,[Backdoor.Oldrea],"[T0814, T0863, T0802]","[User Execution, Automated Collection, Denial ..."


,software_ID,software,technique_ID,technique
0,S0038,[Duqu],"[T0811, T0893]","[Data from Local System, Data from Information..."
1,S0089,[BlackEnergy],[T0865],[Spearphishing Attachment]
2,S0093,[Backdoor.Oldrea],"[T0861, T0888, T0865, T0846, T0862]","[Remote System Discovery, Supply Chain Comprom..."


(21, 4), (21, 4), (17, 4)


#### **Guardado**

In [950]:
if save_as_csv:
    save_df_as_csv(MITRE_techniques_software_NN_df, f'[MITRE]_{matrix}_techniques_software_NN', matrix, 'relations/techniques_software')
    save_df_as_csv(CP_techniques_software_NN_df, f'[CP]_{matrix}_techniques_software_NN', matrix, 'relations/techniques_software')
    save_df_as_csv(NOCP_techniques_software_NN_df, f'[NOCP]_{matrix}_techniques_software_NN', matrix, 'relations/techniques_software')

    save_df_as_csv(MITRE_technique_software_1N_df, f'[MITRE]_{matrix}_technique_software_1N', matrix, 'relations/techniques_software')
    save_df_as_csv(CP_technique_software_1N_df, f'[CP]_{matrix}_technique_software_1N', matrix, 'relations/techniques_software')
    save_df_as_csv(NOCP_technique_software_1N_df, f'[NOCP]_{matrix}_technique_software_1N', matrix, 'relations/techniques_software')

    save_df_as_csv(MITRE_techniques_software_N1_df, f'[MITRE]_{matrix}_techniques_software_N1', matrix, 'relations/techniques_software')
    save_df_as_csv(CP_techniques_software_N1_df, f'[CP]_{matrix}_techniques_software_N1', matrix, 'relations/techniques_software')
    save_df_as_csv(NOCP_techniques_software_N1_df, f'[NOCP]_{matrix}_techniques_software_N1', matrix, 'relations/techniques_software')

Archivo guardado correctamente [MITRE]_ics_techniques_software_NN.csv
Archivo guardado correctamente [CP]_ics_techniques_software_NN.csv
Archivo guardado correctamente [NOCP]_ics_techniques_software_NN.csv
Archivo guardado correctamente [MITRE]_ics_technique_software_1N.csv
Archivo guardado correctamente [CP]_ics_technique_software_1N.csv
Archivo guardado correctamente [NOCP]_ics_technique_software_1N.csv
Archivo guardado correctamente [MITRE]_ics_techniques_software_N1.csv
Archivo guardado correctamente [CP]_ics_techniques_software_N1.csv
Archivo guardado correctamente [NOCP]_ics_techniques_software_N1.csv


### **3.6. Relación grupos - software**

#### **Generación de las tablas**

In [951]:
MITRE_groups_software_NN_df, MITRE_group_software_1N_df, MITRE_groups_software_N1_df = get_groups_software_relationships(mitre_matrix)

Dataframes generados correctamente!


#### **Debug**

In [952]:
if debug_df:
    display(MITRE_groups_software_NN_df.head(3))
    display(MITRE_group_software_1N_df.head(3))
    display(MITRE_groups_software_N1_df.head(3))
    print(f'{MITRE_groups_software_NN_df.shape}, {MITRE_group_software_1N_df.shape}, {MITRE_groups_software_N1_df.shape}')

,group_ID,group,software_ID,software
0,G0034,Sandworm Team,S0606,Bad Rabbit
1,G0032,Lazarus Group,S0366,WannaCry
2,G0046,FIN7,S0496,REvil


,group_ID,group,software_ID,software
0,G0032,[Lazarus Group],[S0366],[WannaCry]
1,G0034,[Sandworm Team],"[S0368, S0604, S0089, S1072, S0607, S0606]","[BlackEnergy, Bad Rabbit, NotPetya, KillDisk, ..."
2,G0035,[Dragonfly],[S0093],[Backdoor.Oldrea]


,software_ID,software,group_ID,group
0,S0089,[BlackEnergy],[G0034],[Sandworm Team]
1,S0093,[Backdoor.Oldrea],[G0035],[Dragonfly]
2,S0366,[WannaCry],[G0032],[Lazarus Group]


(15, 4), (9, 4), (12, 4)


#### **Guardado**

In [953]:
if save_as_csv:
    save_df_as_csv(MITRE_groups_software_NN_df, f'[MITRE]_{matrix}_groups_software_NN', matrix, 'relations/groups_software')
    save_df_as_csv(MITRE_group_software_1N_df, f'[MITRE]_{matrix}_group_software_1N', matrix, 'relations/groups_software')
    save_df_as_csv(MITRE_groups_software_N1_df, f'[MITRE]_{matrix}_groups_software_N1', matrix, 'relations/groups_software')

Archivo guardado correctamente [MITRE]_ics_groups_software_NN.csv
Archivo guardado correctamente [MITRE]_ics_group_software_1N.csv
Archivo guardado correctamente [MITRE]_ics_groups_software_N1.csv


## **4. Disponibilidad de reglas por tipología**

### **4.1. Reglas disponibles por técnica**

In [954]:
CP_detail_rules_techniques, CP_agg_rules_techniques = get_availability_rules_by_technique(matrix)

In [955]:
if debug_df:
    display(CP_detail_rules_techniques.head(3))
    display(CP_agg_rules_techniques.head(3))
    print(f'{CP_detail_rules_techniques.shape}, {CP_agg_rules_techniques.shape}')

,rule,ttp,source,matrix
0,T0802 - Cyberproof - Windows - Log Volume vari...,T0802,UCM Catalog 2024 Sentinel,ics
1,T0802 - Cyberproof - Syslog - Log Volume varia...,T0802,UCM Catalog 2024 Sentinel,ics
2,T0802 - Cyberproof - OKTA - Log Volume varianc...,T0802,UCM Catalog 2024 Sentinel,ics


,ttp,rules,matrix
0,T0858,10,ics
1,T0802,9,ics
2,T0884,9,ics


(106, 4), (32, 3)


In [956]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques, f'{matrix}_detail_rules_techniques', matrix, 'availability_rules/techniques')
    save_df_as_csv(CP_agg_rules_techniques, f'{matrix}_agg_rules_techniques', matrix, 'availability_rules/techniques')

Archivo guardado correctamente ics_detail_rules_techniques.csv
Archivo guardado correctamente ics_agg_rules_techniques.csv


### **4.2. Reglas disponibles por técnica y táctica**

In [957]:
CP_detail_rules_techniques_tactics, CP_agg_rules_techniques_tactics = get_availability_rules_by_technique_and_tactics(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [958]:
if debug_df:
    display(CP_detail_rules_techniques_tactics.head(3))
    display(CP_agg_rules_techniques_tactics.head(3))
    print(f'{CP_detail_rules_techniques_tactics.shape}, {CP_agg_rules_techniques_tactics.shape}')

,tactic_ID,tactic,technique_ID,technique,rules,matrix
0,TA0100,Collection,T0802,Automated Collection,9,ics
1,TA0100,Collection,T0868,Detect Operating Mode,1,ics
2,TA0101,Command and Control,T0884,Connection Proxy,9,ics


,tactic_ID,tactic,techniques,rules,matrix
0,TA0100,Collection,2,10,ics
1,TA0101,Command and Control,3,11,ics
2,TA0102,Discovery,2,4,ics


(37, 6), (12, 5)


In [959]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_tactics, f'{matrix}_detail_rules_techniques_tactics', matrix, 'availability_rules/techniques_tactics')
    save_df_as_csv(CP_agg_rules_techniques_tactics, f'{matrix}_agg_rules_techniques_tactics', matrix, 'availability_rules/techniques_tactics')

Archivo guardado correctamente ics_detail_rules_techniques_tactics.csv
Archivo guardado correctamente ics_agg_rules_techniques_tactics.csv


### **4.3. Reglas disponibles por técnica y data source**

In [960]:
CP_detail_rules_techniques_datasources, CP_agg_rules_techniques_datasources = get_availability_rules_by_technique_and_datasource(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [961]:
if debug_df:
    display(CP_detail_rules_techniques_datasources.head(3))
    display(CP_agg_rules_techniques_datasources.head(3))
    print(f'{CP_detail_rules_techniques_datasources.shape}, {CP_agg_rules_techniques_datasources.shape}')

,data_source_ID,data_source,technique_ID,technique,rules,matrix
0,DS0002,User Account,T0859,Valid Accounts,6,ics
1,DS0009,Process,T0886,Remote Services,4,ics
2,DS0009,Process,T0863,User Execution,5,ics


,data_source_ID,data_source,techniques,rules,matrix
0,DS0002,User Account,1,6,ics
1,DS0009,Process,7,25,ics
2,DS0011,Module,2,12,ics


(74, 6), (13, 5)


In [962]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_datasources, f'{matrix}_detail_rules_techniques_datasources', matrix, 'availability_rules/techniques_datasources')
    save_df_as_csv(CP_agg_rules_techniques_datasources, f'{matrix}_agg_rules_techniques_datasources', matrix, 'availability_rules/techniques_datasources')

Archivo guardado correctamente ics_detail_rules_techniques_datasources.csv
Archivo guardado correctamente ics_agg_rules_techniques_datasources.csv


### **4.4. Reglas disponibles por técnica y plataforma**

In [963]:
CP_detail_rules_techniques_platforms, CP_agg_rules_techniques_platforms = get_availability_rules_by_technique_and_platform(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [964]:
if debug_df:
    display(CP_detail_rules_techniques_platforms.head(3))
    display(CP_agg_rules_techniques_platforms.head(3))
    print(f'{CP_detail_rules_techniques_platforms.shape}, {CP_agg_rules_techniques_platforms.shape}')

,platform,technique_ID,technique,rules,matrix
0,None,T0858,Change Operating Mode,10,ics
1,None,T0827,Loss of Control,1,ics
2,None,T0828,Loss of Productivity and Revenue,1,ics


,platform,techniques,rules,matrix
0,None,32,106,ics


(32, 5), (1, 4)


In [965]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_platforms, f'{matrix}_detail_rules_techniques_platforms', matrix, 'availability_rules/techniques_platforms')
    save_df_as_csv(CP_agg_rules_techniques_platforms, f'{matrix}_agg_rules_techniques_platforms', matrix, 'availability_rules/techniques_platforms')

Archivo guardado correctamente ics_detail_rules_techniques_platforms.csv
Archivo guardado correctamente ics_agg_rules_techniques_platforms.csv


### **4.5. Reglas disponibles por técnica y grupo**

In [966]:
CP_detail_rules_techniques_groups, CP_agg_rules_techniques_groups = get_availability_rules_by_technique_and_group(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [967]:
if debug_df:
    display(CP_detail_rules_techniques_groups.head(3))
    display(CP_agg_rules_techniques_groups.head(3))
    print(f'{CP_detail_rules_techniques_groups.shape}, {CP_agg_rules_techniques_groups.shape}')

,group_ID,group,technique_ID,technique,rules,matrix
0,G0034,Sandworm Team,T0884,Connection Proxy,9,ics
1,G0034,Sandworm Team,T0807,Command-Line Interface,3,ics
2,G0049,OilRig,T0853,Scripting,8,ics


,group_ID,group,techniques,rules,matrix
0,G0034,Sandworm Team,2,12,ics
1,G0049,OilRig,3,15,ics
2,G0064,APT33,1,8,ics


(34, 6), (4, 5)


In [968]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_groups, f'{matrix}_detail_rules_techniques_groups', matrix, 'availability_rules/techniques_groups')
    save_df_as_csv(CP_agg_rules_techniques_groups, f'{matrix}_agg_rules_techniques_groups', matrix, 'availability_rules/techniques_groups')

Archivo guardado correctamente ics_detail_rules_techniques_groups.csv
Archivo guardado correctamente ics_agg_rules_techniques_groups.csv


### **4.6. Reglas disponibles por técnica y software**

In [969]:
CP_detail_rules_techniques_software, CP_agg_rules_techniques_software = get_availability_rules_by_technique_and_software(matrix, mitre_matrix, cp_techniques)

Dataframes generados correctamente!


In [970]:
if debug_df:
    display(CP_detail_rules_techniques_software.head(3))
    display(CP_agg_rules_techniques_software.head(3))
    print(f'{CP_detail_rules_techniques_software.shape}, {CP_agg_rules_techniques_software.shape}')

,software_ID,software,technique_ID,technique,rules,matrix
0,S0038,Duqu,T0882,Theft of Operational Information,3,ics
1,S0089,BlackEnergy,T0869,Standard Application Layer Protocol,1,ics
2,S0089,BlackEnergy,T0859,Valid Accounts,6,ics


,software_ID,software,techniques,rules,matrix
0,S0038,Duqu,1,3,ics
1,S0089,BlackEnergy,2,7,ics
2,S0093,Backdoor.Oldrea,3,17,ics


(80, 6), (21, 5)


In [971]:
if save_as_csv:
    save_df_as_csv(CP_detail_rules_techniques_software, f'{matrix}_detail_rules_techniques_software', matrix, 'availability_rules/techniques_software')
    save_df_as_csv(CP_agg_rules_techniques_software, f'{matrix}_agg_rules_techniques_software', matrix, 'availability_rules/techniques_software')

Archivo guardado correctamente ics_detail_rules_techniques_software.csv
Archivo guardado correctamente ics_agg_rules_techniques_software.csv
